In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("atp_matches_2024.csv")
df

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2024-0339,Brisbane,Hard,32,A,20240101,300,105777,2.0,NaN,...,58.0,44.0,16.0,11.0,8.0,9.0,14.0,2570.0,8.0,3660.0
1,2024-0339,Brisbane,Hard,32,A,20240101,299,208029,1.0,NaN,...,35.0,31.0,10.0,11.0,5.0,7.0,8.0,3660.0,39.0,1122.0
2,2024-0339,Brisbane,Hard,32,A,20240101,298,105777,2.0,NaN,...,39.0,24.0,14.0,10.0,5.0,7.0,14.0,2570.0,55.0,902.0
3,2024-0339,Brisbane,Hard,32,A,20240101,297,208029,1.0,NaN,...,51.0,31.0,16.0,10.0,3.0,5.0,8.0,3660.0,116.0,573.0
4,2024-0339,Brisbane,Hard,32,A,20240101,296,126128,NaN,NaN,...,37.0,27.0,16.0,10.0,5.0,8.0,39.0,1122.0,44.0,1021.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3071,2024-M-DC-2024-WG2-PO-URU-MDA-01,Davis Cup WG2 PO: URU vs MDA,Clay,4,D,20240203,5,212051,NaN,NaN,...,30.0,17.0,7.0,6.0,8.0,14.0,1109.0,8.0,740.0,34.0
3072,2024-M-DC-2024-WG2-PO-VIE-RSA-01,Davis Cup WG2 PO: VIE vs RSA,Hard,4,D,20240202,1,122533,NaN,NaN,...,41.0,25.0,6.0,9.0,1.0,4.0,554.0,67.0,748.0,32.0
3073,2024-M-DC-2024-WG2-PO-VIE-RSA-01,Davis Cup WG2 PO: VIE vs RSA,Hard,4,D,20240202,2,144748,NaN,NaN,...,51.0,25.0,7.0,11.0,5.0,12.0,416.0,109.0,NaN,NaN
3074,2024-M-DC-2024-WG2-PO-VIE-RSA-01,Davis Cup WG2 PO: VIE vs RSA,Hard,4,D,20240202,4,122533,NaN,NaN,...,51.0,32.0,17.0,14.0,5.0,9.0,554.0,67.0,416.0,109.0


In [5]:
# Step 1: Create player-level rows for both winner and loser
winner_df = df.rename(columns={
    'winner_name': 'player_name',
    'w_ace': 'ace',
    'w_df': 'df',
    'w_1stWon': 'first_serve_won',
    'w_2ndWon': 'second_serve_won',
    'w_bpSaved': 'bp_saved',
    'w_svpt': 'serve_pts',
    'surface': 'surface'
})
loser_df = df.rename(columns={
    'loser_name': 'player_name',
    'l_ace': 'ace',
    'l_df': 'df',
    'l_1stWon': 'first_serve_won',
    'l_2ndWon': 'second_serve_won',
    'l_bpSaved': 'bp_saved',
    'l_svpt': 'serve_pts',
    'surface': 'surface'
})

# Add a 'player_result' column if needed
winner_df['player_result'] = 1
loser_df['player_result'] = 0

# Combine both
player_df = pd.concat([winner_df[['player_name', 'surface', 'ace', 'df', 'first_serve_won',
                                  'second_serve_won', 'bp_saved', 'serve_pts']],
                       loser_df[['player_name', 'surface', 'ace', 'df', 'first_serve_won',
                                 'second_serve_won', 'bp_saved', 'serve_pts']]], ignore_index=True)


In [6]:
player_df

,player_name,surface,ace,df,first_serve_won,second_serve_won,bp_saved,serve_pts
0,Grigor Dimitrov,Hard,8.0,2.0,40.0,13.0,3.0,74.0
1,Holger Rune,Hard,7.0,4.0,39.0,11.0,1.0,72.0
2,Grigor Dimitrov,Hard,10.0,3.0,39.0,10.0,6.0,67.0
3,Holger Rune,Hard,13.0,0.0,31.0,17.0,1.0,65.0
4,Roman Safiullin,Hard,9.0,3.0,36.0,14.0,2.0,73.0
...,...,...,...,...,...,...,...,...
6147,Ilya Snitari,Clay,1.0,1.0,17.0,7.0,8.0,61.0
6148,Philip Henning,Hard,2.0,1.0,25.0,6.0,1.0,56.0
6149,Linh Giang Trinh,Hard,0.0,2.0,25.0,7.0,5.0,71.0
6150,Kris Van Wyk,Hard,5.0,3.0,32.0,17.0,5.0,86.0


In [7]:
# Avoid division by zero
player_df = player_df[player_df['serve_pts'] > 0]

# Compute rates
player_df['ace_rate'] = player_df['ace'] / player_df['serve_pts']
player_df['df_rate'] = player_df['df'] / player_df['serve_pts']
player_df['first_serve_win_pct'] = player_df['first_serve_won'] / player_df['serve_pts']
player_df['second_serve_win_pct'] = player_df['second_serve_won'] / player_df['serve_pts']
player_df['bp_saved_pct'] = player_df['bp_saved'] / player_df['serve_pts']

# Aggregate per player per surface
player_surface_stats = player_df.groupby(['player_name', 'surface']).agg({
    'ace_rate': 'mean',
    'df_rate': 'mean',
    'first_serve_win_pct': 'mean',
    'second_serve_win_pct': 'mean',
    'bp_saved_pct': 'mean'
}).reset_index()
player_surface_stats

/var/folders/3t/356b2m1s0713vmrmjlw72vfm0000gn/T/ipykernel_51869/4126321600.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['ace_rate'] = player_df['ace'] / player_df['serve_pts']
/var/folders/3t/356b2m1s0713vmrmjlw72vfm0000gn/T/ipykernel_51869/4126321600.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['df_rate'] = player_df['df'] / player_df['serve_pts']
/var/folders/3t/356b2m1s0713vmrmjlw72vfm0000gn/T/ipykernel_51869/4126321600.py:7: SettingWithCopyWarning: 
A value is trying t

,player_name,surface,ace_rate,df_rate,first_serve_win_pct,second_serve_win_pct,bp_saved_pct
0,Abedallah Shelbayh,Clay,0.018519,0.018519,0.370370,0.166667,0.092593
1,Abedallah Shelbayh,Hard,0.045625,0.080417,0.373542,0.188542,0.071250
2,Adam Neff,Hard,0.055556,0.111111,0.333333,0.166667,0.097222
3,Adam Walton,Clay,0.009259,0.009259,0.379630,0.185185,0.120370
4,Adam Walton,Grass,0.044057,0.026941,0.426104,0.219971,0.040989
...,...,...,...,...,...,...,...
779,Zizou Bergs,Clay,0.089277,0.033331,0.458364,0.183916,0.059707
780,Zizou Bergs,Grass,0.031778,0.026362,0.499484,0.172811,0.048891
781,Zizou Bergs,Hard,0.092297,0.038668,0.449385,0.203865,0.048503
782,Zsombor Piros,Hard,0.100000,0.014286,0.428571,0.228571,0.057143


In [8]:
from sklearn.preprocessing import StandardScaler

normalized_data = []
scalers = {}

for surface in player_surface_stats['surface'].unique():
    temp = player_surface_stats[player_surface_stats['surface'] == surface].copy()
    features = temp.drop(columns=['player_name', 'surface'])
    
    scaler = StandardScaler()
    temp_scaled = scaler.fit_transform(features)
    scalers[surface] = scaler  # Save if needed later

    temp[features.columns] = temp_scaled
    normalized_data.append(temp)

surface_normalized = pd.concat(normalized_data, ignore_index=True)


In [10]:
from sklearn.cluster import KMeans

clusters = []
kmeans_models = {}

for surface in surface_normalized['surface'].unique():
    temp = surface_normalized[surface_normalized['surface'] == surface].copy()
    features = temp.drop(columns=['player_name', 'surface'])

    kmeans = KMeans(n_clusters=3, random_state=42)
    temp['cluster'] = kmeans.fit_predict(features)

    kmeans_models[surface] = kmeans
    clusters.append(temp)

surface_clustered = pd.concat(clusters, ignore_index=True)
surface_clustered

,player_name,surface,ace_rate,df_rate,first_serve_win_pct,second_serve_win_pct,bp_saved_pct,cluster
0,Abedallah Shelbayh,Clay,-0.951206,-0.797930,-0.765878,-0.418958,1.395204,0
1,Adam Walton,Clay,-1.228732,-1.166678,-0.602696,0.012267,2.476751,0
2,Adria Soriano Barrera,Clay,1.556353,1.130859,0.789941,0.191091,-0.108856,2
3,Adrian Mannarino,Clay,-0.288443,-0.996426,-1.947621,0.790925,0.053032,1
4,Albert Ramos,Clay,-1.091789,-0.508138,-0.645089,-0.520683,0.111077,1
...,...,...,...,...,...,...,...,...
779,Yannick Hanfmann,Grass,0.151874,-0.428224,1.329028,-1.907504,-0.918810,1
780,Yoshihito Nishioka,Grass,-0.983811,-0.812949,-0.141095,0.753793,0.038600,0
781,Zachary Svajda,Grass,-1.475682,1.386319,0.178696,-1.454514,-0.828015,2
782,Zhizhen Zhang,Grass,0.778052,-0.472830,0.720654,-0.330264,-1.366170,1


In [12]:
surface_clustered[surface_clustered['player_name'] == 'Abedallah Shelbayh']

,player_name,surface,ace_rate,df_rate,first_serve_win_pct,second_serve_win_pct,bp_saved_pct,cluster
0,Abedallah Shelbayh,Clay,-0.273423,0.986769,-0.719158,-0.356429,0.754021,0
213,Abedallah Shelbayh,Grass,-0.969397,1.428726,-1.798892,0.311422,0.480164,1
363,Abedallah Shelbayh,Hard,0.336507,0.119436,-0.075152,0.100919,0.308396,0


In [11]:
# See player style clusters across surfaces
surface_clustered[['player_name', 'surface', 'cluster']].sort_values(by=['player_name', 'surface']).to_csv("surface_clusters_2024.csv")
